In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.linalg import expm
from math import comb, atan2
from itertools import product

In [4]:
def tdhf(t,y):
    th, phi = y
    d_th = -1*np.sin(phi)
    d_phi = np.tan(th)*np.cos(phi)+mu*np.sin(th)
    return [d_th, d_phi]

def tdhf_integrate(t_list,y0):
    sol = solve_ivp(tdhf, 
                                    (t_list[0],t_list[-1]), 
                                    y0, 
                                    t_eval = t_list, 
                                    method='Radau') 
    # LSODA and BDF methods were unstable and not conserving energy, Radau is much better
    # RK45 is a non-stiff method, even worse
    if not sol.success:
        raise RuntimeError(sol.message)
    return sol

def make_y0(HFmin,allL):
    if HFmin:
        th = atan2(-1/mu,-np.sqrt(mu**2-1)/mu) 
        phi = 0
    if allL:
        th = np.pi/2
        phi = 0
    if allL == HFmin:
        raise AttributeError('pick a starting state queen')
    return th, phi

def LR(th,phi):
    L = np.sqrt( (1+np.sin(th))/2 )
    R = np.sqrt( (1-np.sin(th))/2 )*np.exp(1j*phi)
    return L,R

In [13]:
mu = -4

P = 4
X = 2

# make_y0(HFmin,allL)
y0 = make_y0(False,True)
t_list = np.linspace(0.0, 100.0, 801) 
sol_tdhf = tdhf_integrate(t_list,y0)

th,phi = sol_tdhf.y[0],sol_tdhf.y[1]
L,R = LR(th,phi)

LRstate = np.array([L,R])
print(LRstate.shape) # (X,t)

manybodystate = LRstate #.transpose(1,0)
print(manybodystate.shape)
for p in range(P):
    manybodystate = np.einsum('yt,xt->yxt',manybodystate,LRstate)
    manybodystate = manybodystate.reshape(X**(p+2),len(t_list))
print(manybodystate.shape)


(2, 801)
(2, 801)
(32, 801)
